# Head-Fixed Cross-Dataset Experiment — Pixel Error vs Ensemble Std Dev

Same plotting machinery as the LP sweep notebook.
The ensemble std dev is computed across seeds *within* each training condition,
then compared across conditions on the same axes.

Set `eval_dataset` in the configuration cell to switch between facemap / ibl-paw.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
import seaborn as sns

## Utility functions

Nearly identical to the reference notebook; two small additions:
- `compute_ensemble_stddev` accepts an optional `keypoints` list to filter predictions
  (needed for the mixed model, whose predictions include keypoints from both datasets).
- `build_ensemble` accepts an optional `error_csv_dict` (keyed by model label) because
  our pixel error CSVs live in `eval/<dataset>/` rather than next to the prediction CSV.

In [ ]:
def standardize_scorer_level(df, new_scorer='standard_scorer'):
    df.columns = pd.MultiIndex.from_tuples(
        [(new_scorer, bodypart, coord) for _, bodypart, coord in df.columns],
        names=df.columns.names,
    )
    return df


def compute_ensemble_stddev(df_preds, scorer_name='standard_scorer', keypoints=None):
    """Compute per-frame, per-keypoint ensemble std dev across models.
    
    keypoints: optional list of bodypart names to include (filters by name).
        Used to restrict mixed-model predictions to the eval dataset's keypoints.
    """
    preds = []
    for i, df in enumerate(df_preds):
        df = standardize_scorer_level(df, scorer_name)

        cols_to_keep = [
            col for col in df.columns
            if not col[2].endswith('likelihood') and 'Unnamed' not in col[2]
        ]
        if keypoints is not None:
            cols_to_keep = [col for col in cols_to_keep if col[1] in keypoints]
        df = df[cols_to_keep]

        if df.isna().any().any():
            nan_indices = df[df.isna().any(axis=1)].index
            nan_columns = df.columns[df.isna().any()]
            print(f'Warning: NaN values in DataFrame {i} — indices: {nan_indices}, columns: {nan_columns}')

        arr_np = df.to_numpy()
        try:
            arr = arr_np.reshape(df.shape[0], -1, 2)
        except ValueError as e:
            print(f'Reshape error: {e}  shape={arr_np.shape}')
            raise
        preds.append(arr)

    preds = np.stack(preds, axis=-1)  # (n_frames, n_keypoints, 2, n_models)

    if np.isnan(preds).any():
        print(f'Warning: NaN values in stacked preds array')
    else:
        print('No NaN values in preds array.')

    return np.std(preds, axis=-1).mean(axis=-1)  # (n_frames, n_keypoints)


def compute_percentiles(arr, std_vals, percentiles):
    num_pts = arr[0]
    vals, prctiles = [], []
    for p in percentiles:
        idx = np.argmin(np.abs(arr - num_pts * p / 100))
        achieved = arr[idx] / num_pts * 100 if idx == len(arr) - 1 else p
        vals.append(std_vals[idx])
        prctiles.append(np.round(achieved, 2))
    return vals, prctiles


class Ensemble:
    data_to_plot: dict[str, str]  # model label -> path to predictions CSV
    pred_csv_list: list[str]
    error_csv_list: list[str]
    df_pred_list: list[pd.DataFrame]
    df_error_list: list[pd.DataFrame]
    n_points_dict: dict[str, dict[str, int]]
    std_vals: list[float]
    df_w_vars: pd.DataFrame
    df_line2: pd.DataFrame


def build_ensemble(ens, std_val_max=8.0, error_csv_dict=None, keypoints=None):
    """Load predictions and pixel errors, compute ensemble statistics.
    
    error_csv_dict: optional dict mapping each key in ens.data_to_plot to an
        explicit pixel-error CSV path. If None, derives path from pred CSV stem
        (original behaviour: predictions_test.csv -> predictions_pixel_error_test.csv).
    keypoints: optional list of bodypart names passed to compute_ensemble_stddev.
    """
    model_names_list = list(ens.data_to_plot.keys())
    pred_csv_list = [Path(v) for v in ens.data_to_plot.values()]

    if error_csv_dict is not None:
        error_csv_list = [Path(error_csv_dict[k]) for k in model_names_list]
    else:
        error_csv_list = [
            p.with_stem(
                p.stem
                .replace('_new', '_pixel_error_new')
                .replace('_test', '_pixel_error_test')
            )
            for p in pred_csv_list
        ]

    df_pred_list, df_error_list = [], []
    for pred_csv, error_csv in zip(pred_csv_list, error_csv_list):
        df_pred_list.append(pd.read_csv(pred_csv, header=[0, 1, 2], index_col=0).sort_index())
        df = pd.read_csv(error_csv, header=[0], index_col=0).sort_index()
        if 'set' in df.columns:
            df = df.drop(columns=['set'])
        df_error_list.append(df)

    ens_stddev = compute_ensemble_stddev(df_pred_list, keypoints=keypoints)

    df_w_vars = []
    for df_error, df_pred, model_name in zip(df_error_list, df_pred_list, model_names_list):
        assert (df_error.index == df_pred_list[0].index).all()
        assert (df_pred.index == df_pred_list[0].index).all()

        print(f'Total pixel error for model {model_name}: {df_error.sum().sum()}')

        for i, kp in enumerate(df_error.columns):
            index = [f'{frame}_{model_name}_{kp}' for frame in df_error.index]
            df_pred_std = standardize_scorer_level(df_pred.copy())
            df_w_vars.append(pd.DataFrame(
                {
                    'pixel_error': df_error[kp].values,
                    'likelihood': df_pred_std.loc[:, ('standard_scorer', kp, 'likelihood')].values,
                    'ens-std': ens_stddev[:, i],
                    'ens-std-prctile': [
                        np.sum(ens_stddev < p) / ens_stddev.size for p in ens_stddev[:, i]
                    ],
                    'ens-std-prctile-kp': [
                        np.sum(ens_stddev[:, i] < p) / ens_stddev[:, i].size for p in ens_stddev[:, i]
                    ],
                    'keypoint': kp,
                    'model': model_name,
                },
                index=index,
            ))

    df_w_vars = pd.concat(df_w_vars)
    std_vals = np.arange(0, std_val_max, 0.25)
    n_points_dict = {m: np.nan * np.zeros_like(std_vals) for m in model_names_list}

    df_line2 = []
    for s, std in enumerate(std_vals):
        df_tmp_ = df_w_vars[df_w_vars['ens-std'] > std]
        for model_name in model_names_list:
            d = df_tmp_[df_tmp_.model == model_name]
            n_points_dict[model_name][s] = np.sum(~d['pixel_error'].isna())
            df_line2.append(pd.DataFrame(
                {
                    'ens-std': std,
                    'model': model_name,
                    'pixel_error': d.pixel_error.to_numpy(),
                    'keypoint': d['keypoint'].to_numpy(),
                },
                index=[f'{row}_{s}' for row in d.index],
            ))

    ens.pred_csv_list = pred_csv_list
    ens.error_csv_list = error_csv_list
    ens.df_pred_list = df_pred_list
    ens.df_error_list = df_error_list
    ens.n_points_dict = n_points_dict
    ens.std_vals = std_vals
    ens.df_w_vars = df_w_vars
    ens.df_line2 = pd.concat(df_line2)
    return ens

## Configuration

Edit the variables below. Set `eval_dataset` to `'facemap'` or `'ibl-paw'` to
switch which held-out test set is evaluated.

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────────
import sys as _sys; _sys.path.insert(0, str(Path('__file__').parents[1] if '__file__' in dir() else Path().resolve().parent))
from mouse_pose.paths import load_paths as _load_paths
BASE_SAVE_DIR = Path(_load_paths()['results_dir'])

# ── eval target ────────────────────────────────────────────────────────────────
# eval_dataset = 'ibl-paw'   # 'facemap' | 'ibl-paw' | 'cheese-2d'
# eval_dataset = 'facemap'
eval_dataset = 'cheese-2d'

# ── sweep parameters ───────────────────────────────────────────────────────────
backbone = 'vits_dino'
losses   = 'supervised'
seeds    = [0, 1, 2]

# ── model conditions ───────────────────────────────────────────────────────────
# csv_stem matches the results directory name produced by train_sweep.py
# (i.e. the portion after 'CollectedData_' in the CSV filename)
model_conditions = {
    'facemap': [
        {'label': 'facemap-200',         'csv_stem': 'facemap-600_train',     'train_frames': 200},
        # {'label': 'facemap-400',         'csv_stem': 'facemap-600_train',     'train_frames': 400},
        {'label': 'facemap-600',         'csv_stem': 'facemap-600_train',     'train_frames': 600},
        {'label': 'face+ibl-400',        'csv_stem': 'face+ibl_train',        'train_frames': 400},
        {'label': 'face+ibl-600',        'csv_stem': 'face+ibl_train',        'train_frames': 600},
        {'label': 'face+cheese-400',     'csv_stem': 'face+cheese_train',     'train_frames': 400},
        {'label': 'face+cheese-600',     'csv_stem': 'face+cheese_train',     'train_frames': 600},
        {'label': 'face+ibl+cheese-600', 'csv_stem': 'face+ibl+cheese_train', 'train_frames': 600},
    ],
    'ibl-paw': [
        {'label': 'ibl-200',             'csv_stem': 'ibl-paw-600_train',     'train_frames': 200},
        # {'label': 'ibl-400',             'csv_stem': 'ibl-paw-600_train',     'train_frames': 400},
        {'label': 'ibl-600',             'csv_stem': 'ibl-paw-600_train',     'train_frames': 600},
        {'label': 'face+ibl-400',        'csv_stem': 'face+ibl_train',        'train_frames': 400},
        {'label': 'face+ibl-600',        'csv_stem': 'face+ibl_train',        'train_frames': 600},
        {'label': 'cheese+ibl-400',      'csv_stem': 'cheese+ibl_train',      'train_frames': 400},
        {'label': 'cheese+ibl-600',      'csv_stem': 'cheese+ibl_train',      'train_frames': 600},
        {'label': 'face+ibl+cheese-600', 'csv_stem': 'face+ibl+cheese_train', 'train_frames': 600},
    ],
    'cheese-2d': [
        {'label': 'cheese-200',          'csv_stem': 'cheese-2d-600_train',   'train_frames': 200},
        # {'label': 'cheese-400',          'csv_stem': 'cheese-2d-600_train',   'train_frames': 400},
        {'label': 'cheese-600',          'csv_stem': 'cheese-2d-600_train',   'train_frames': 600},
        {'label': 'cheese+ibl-400',      'csv_stem': 'cheese+ibl_train',      'train_frames': 400},
        {'label': 'cheese+ibl-600',      'csv_stem': 'cheese+ibl_train',      'train_frames': 600},
        {'label': 'face+cheese-400',     'csv_stem': 'face+cheese_train',     'train_frames': 400},
        {'label': 'face+cheese-600',     'csv_stem': 'face+cheese_train',     'train_frames': 600},
        {'label': 'face+ibl+cheese-600', 'csv_stem': 'face+ibl+cheese_train', 'train_frames': 600},
    ],
}

# y-axis limit for the main plot
ylim_map = {
    'facemap':   80,
    'ibl-paw':   40,
    'cheese-2d': 100,
}

# one color per model condition entry, in order
colors = [
    '#a8c8f0',  # light blue   — ibl-200
    # '#4a90d9',  # medium blue  — ibl-400
    '#034e96',  # dark blue    — ibl-600
    '#e05b28',  # orange       — face+ibl-400
    '#8c3b1c',  # dark oragne  — face+ibl-600
    '#08961d',  # green        — cheese+ibl-400
    '#0d6119',  # dark green   — cheese+ibl-600
    # '#d152be',  # pink
    '#c0392b',  # red          — face+ibl+cheese-600
]

# ── build ens.data_to_plot and error_csv_dict ──────────────────────────────────
def _sanitize(s):
    return s.replace('/', '_').replace('.', '_')

conditions = model_conditions[eval_dataset]

ens = Ensemble()
ens.data_to_plot = {}
error_csv_dict   = {}

for cond in conditions:
    for seed in seeds:
        key = f"{cond['label']}.{seed}"
        model_dir = (
            BASE_SAVE_DIR / cond['csv_stem'] / losses
            / f"tf{cond['train_frames']}" / _sanitize(backbone)
            / f'seed{seed}'
        )
        ens.data_to_plot[key] = str(model_dir / 'eval' / eval_dataset / 'predictions.csv')
        error_csv_dict[key]   = str(model_dir / 'eval' / eval_dataset / 'pixel_error.csv')

build_ensemble(ens, error_csv_dict=error_csv_dict)

models_to_plot = [c['label'] for c in conditions]
color_mapping  = {m: c for m, c in zip(models_to_plot, colors)}
dataset_name   = eval_dataset

def major_model(s):
    return '.'.join(s.split('.')[:-1])
ens.df_line2['model2'] = ens.df_line2['model'].apply(major_model)

## Main plot

In [ ]:
df_line2      = ens.df_line2
n_points_dict = ens.n_points_dict
std_vals      = ens.std_vals

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for model in models_to_plot:
    data = df_line2[df_line2['model2'] == model].reset_index(drop=True)
    sns.lineplot(x='ens-std', y='pixel_error', data=data, label=model,
                 color=color_mapping[model], ax=ax, errorbar='se', linewidth=2)

ax.set_title(f'{dataset_name}', fontsize=16)
ax.set_ylabel('Pixel error', fontsize=14)
ax.set_xlabel('Ensemble std dev', fontsize=14)
ax.set_ylim(bottom=0)
if dataset_name in ylim_map:
    ax.set_ylim(top=ylim_map[dataset_name])
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=12)

# percentile annotations based on first model's first seed
percentiles = [95, 50, 5]
vals, prctiles = compute_percentiles(
    arr=n_points_dict[f'{models_to_plot[0]}.0'],
    std_vals=std_vals,
    percentiles=percentiles,
)
for p, v in zip(prctiles, vals):
    ax.axvline(v, ymax=0.95, linestyle='--', linewidth=1, color='black', alpha=0.5)
    ax.text(v, ax.get_ylim()[1], f'{p}%', ha='left', va='top', fontsize=10, rotation=90)

plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Calculate column widths
model_width = max(len(model) for model in models_to_plot)
val_width = 8  # Adjust based on your data precision needs

# Print header
print('|' + ' ' * model_width + '|', end='')
for val in prctiles:
    print(f'{val:^{val_width}}|', end='')
print()

# Print separator line
print('|' + '-' * model_width + '|', end='')
for val in prctiles:
    print('-' * val_width + '|', end='')
print()

# Print data rows
for model in models_to_plot:
    print(f'|{model:<{model_width}}|', end='')
    for val in vals:
        err = df_line2[(df_line2['ens-std']==val) & (df_line2['model2']==model)].pixel_error.mean()
        print(f'{round(err, 2):^{val_width}}|', end='')
    print()

# Print bottom border
print('|' + '-' * model_width + '|', end='')
for val in vals:
    print('-' * val_width + '|', end='')
print()

In [ ]:
keypoints = ens.df_error_list[0].columns.tolist()
ncols = 5
nrows = int(np.ceil(len(keypoints) / 4))
fig, axes = plt.subplots(
    ncols=ncols,
    nrows=nrows,
    figsize=(3 * ncols, 3.5 * nrows),
)
axes = axes.flatten()
for ax in axes:
    ax.axis('off')

#models_to_plot = list(ens.data_to_plot.keys())
df_line2 = ens.df_line2
n_points_dict = ens.n_points_dict
std_vals = ens.std_vals

for j, keypoint in enumerate(keypoints):
    ax = axes[j]
    ax.axis('on')
    for model in models_to_plot:
        data = df_line2[
            (df_line2["model2"] == model) & (df_line2["keypoint"] == keypoint)
        ].reset_index(drop=True)
        sns.lineplot(
            x="ens-std",
            y="pixel_error",
            data=data,
            label=model,
            color=color_mapping[model],
            ax=ax,
            errorbar="se",
            linewidth=2,
        )

    if j % ncols == 0:
        ax.set_ylabel(f"Pixel error", fontsize=12)
    else:
        ax.set_ylabel('')
    if (j // nrows) == nrows - 1:
        ax.set_xlabel('Ens s.d.', fontsize=12)
    else:
        ax.set_xlabel('')
    ax.set_title(keypoint, fontsize=12)
    ax.tick_params(axis="both", which="major", labelsize=10)

    # Display legend only for right hand side plot.
    if j != 0:
        ax.legend().set_visible(False)
        # ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8, frameon=False)

    # Plot annotations
    percentiles = [95, 50, 5]
    vals, prctiles = compute_percentiles(
        arr=n_points_dict[models_to_plot[0] + '.0'],
        std_vals=std_vals,
        percentiles=percentiles,
    )

    for p, v in zip(prctiles, vals):
        ax.axvline(
            v, ymax=0.95, linestyle="--", linewidth=1, color="black", alpha=0.5
        )
        ax.text(
            v,
            ax.get_ylim()[1],
            f"{p}%",
            ha="left",
            va="top",
            fontsize=8,
            rotation=90,
        )

    ax.grid(True, alpha=0.3)

plt.subplots_adjust(bottom=0.25, hspace=0.8)
plt.show()